<a href="https://colab.research.google.com/github/aycaaozturk/AML-project/blob/main/AML_clinical_patient_normalize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# AML CLINICAL PATIENT DATA
# CATEGORY CLEANING + SKEWNESS HANDLING
# + ENCODING + SCALING FOR SURVIVAL MODELS
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


# ------------------------------------------------------------
# 1. File paths
# ------------------------------------------------------------

INPUT_DIR = Path(
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/"
    "Output Files 2/Clinical Patient/split_and_imputed"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/"
    "Output Files 2/Clinical Patient/survival_model_ready"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = INPUT_DIR / "train_imputed.csv"
VAL_PATH = INPUT_DIR / "validation_imputed.csv"
TEST_PATH = INPUT_DIR / "test_imputed.csv"


# ------------------------------------------------------------
# 2. Modeling settings
# ------------------------------------------------------------

ID_COL = "PATIENT_ID"
TIME_COL = "OS_DAYS"
EVENT_COL = "OS_EVENT"

# Risk group contains prognostic information.
# True: allow the model to use the existing clinical risk classification.
# False: assess prediction without the pre-existing risk classification.
INCLUDE_RISK_GROUP = True

# Protocol may represent treatment and study-era differences.
INCLUDE_PROTOCOL = False

# Transform WBC because it is strongly right-skewed.
APPLY_WBC_LOG_TRANSFORMATION = True

# ------------------------------------------------------------
# 3. Load train, validation and test datasets
# ------------------------------------------------------------

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Original dataset shapes")
print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)



Original dataset shapes
Training: (620, 22)
Validation: (177, 22)
Test: (89, 22)


In [ ]:
# ------------------------------------------------------------
# 4. Basic dataset validation
# ------------------------------------------------------------

required_cols = [
    ID_COL,
    TIME_COL,
    EVENT_COL,
    "RISK_GROUP",
    "CYTOGENETIC_COMPLEXITY",
    "WBC"
]

for split_name, split_df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    missing_required = [
        col for col in required_cols
        if col not in split_df.columns
    ]

    if missing_required:
        raise KeyError(
            f"{split_name} is missing required columns: "
            f"{missing_required}"
        )

    if split_df[ID_COL].duplicated().any():
        raise ValueError(
            f"{split_name} contains duplicate patient IDs."
        )


# Confirm that no patient occurs in multiple sets.
train_ids = set(train_df[ID_COL].astype(str))
val_ids = set(val_df[ID_COL].astype(str))
test_ids = set(test_df[ID_COL].astype(str))

assert train_ids.isdisjoint(val_ids), \
    "Patient overlap between training and validation."

assert train_ids.isdisjoint(test_ids), \
    "Patient overlap between training and test."

assert val_ids.isdisjoint(test_ids), \
    "Patient overlap between validation and test."

In [ ]:
# ------------------------------------------------------------
# 5. Validate survival outcomes
# ------------------------------------------------------------

for split_name, split_df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    if split_df[TIME_COL].isna().any():
        raise ValueError(
            f"{split_name} has missing {TIME_COL} values."
        )

    if split_df[EVENT_COL].isna().any():
        raise ValueError(
            f"{split_name} has missing {EVENT_COL} values."
        )

    if (split_df[TIME_COL] < 0).any():
        raise ValueError(
            f"{split_name} has negative survival times."
        )

    invalid_events = set(
        split_df[EVENT_COL].dropna().unique()
    ) - {0, 1, 0.0, 1.0}

    if invalid_events:
        raise ValueError(
            f"{split_name} has invalid event values: "
            f"{invalid_events}"
        )


# ------------------------------------------------------------
# 6. Clean text formatting
# ------------------------------------------------------------

def strip_text_values(dataframe):
    """
    Remove unnecessary spaces from string/category columns.
    """
    dataframe = dataframe.copy()

    text_cols = dataframe.select_dtypes(
        include=["object", "category"]
    ).columns

    for col in text_cols:
        dataframe[col] = dataframe[col].apply(
            lambda value:
            value.strip()
            if isinstance(value, str)
            else value
        )

    return dataframe


train_df = strip_text_values(train_df)
val_df = strip_text_values(val_df)
test_df = strip_text_values(test_df)


In [ ]:
# ------------------------------------------------------------
# 7. Clean RISK_GROUP
# ------------------------------------------------------------

def clean_risk_group(series):
    """
    Standardize risk-group labels.

    Confirmed mapping:
        10 -> Low
        30 -> High

    Missing values are retained as an explicit 'Missing' category
    instead of assigning an unsupported clinical risk group.
    """

    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    risk_mapping = {
        "10": "Low",
        "low": "Low",

        "standard": "Standard",

        "30": "High",
        "high": "High",

        # Common possible formatting variants:
        "low risk": "Low",
        "standard risk": "Standard",
        "high risk": "High"
    }

    cleaned = normalized.map(risk_mapping)

    # Original missing values become <NA> after astype("string").
    cleaned = cleaned.fillna("Missing")

    return cleaned


for dataframe in [train_df, val_df, test_df]:
    dataframe["RISK_GROUP_CLEAN"] = clean_risk_group(
        dataframe["RISK_GROUP"]
    )


print("\nCleaned training risk groups:")
print(
    train_df["RISK_GROUP_CLEAN"]
    .value_counts(dropna=False)
)


# Check whether any original non-missing values were not recognized.
def show_unrecognized_risk_values(dataframe, split_name):
    original = dataframe["RISK_GROUP"].astype("string").str.strip()

    recognized_values = {
        "10", "30",
        "Low", "Standard", "High",
        "low", "standard", "high",
        "Low Risk", "Standard Risk", "High Risk",
        "low risk", "standard risk", "high risk"
    }

    unexpected = original[
        original.notna() &
        ~original.isin(recognized_values)
    ].unique()

    if len(unexpected) > 0:
        print(
            f"Warning: unrecognized RISK_GROUP values "
            f"in {split_name}: {unexpected}"
        )


show_unrecognized_risk_values(train_df, "training")
show_unrecognized_risk_values(val_df, "validation")
show_unrecognized_risk_values(test_df, "test")



Cleaned training risk groups:
RISK_GROUP_CLEAN
Standard    278
Low         239
High         80
Missing      23
Name: count, dtype: int64


In [ ]:
# ------------------------------------------------------------
# 8. Clean CYTOGENETIC_COMPLEXITY
# ------------------------------------------------------------

def clean_cytogenetic_complexity(series):
    """
    Convert cytogenetic-complexity entries into clinically
    interpretable ordinal categories:

        0
        1
        2
        3
        4_plus

    Values 4, 5, 6, 10, '4 or more', and 'More than 6'
    are all grouped as 4_plus.

    This is preferable to assigning an arbitrary exact number to
    top-coded descriptions such as 'More than 6'.
    """

    text = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    def map_value(value):
        if pd.isna(value):
            return "Missing"

        # Explicit text categories.
        if value in {
            "4 or more",
            "more than 6",
            ">6",
            ">=4",
            "4+"
        }:
            return "4_plus"

        # Try converting ordinary numeric strings.
        try:
            numeric_value = float(value)

            if numeric_value == 0:
                return "0"
            elif numeric_value == 1:
                return "1"
            elif numeric_value == 2:
                return "2"
            elif numeric_value == 3:
                return "3"
            elif numeric_value >= 4:
                return "4_plus"

        except (TypeError, ValueError):
            return "Unrecognized"

        return "Unrecognized"

    return text.apply(map_value)


for dataframe in [train_df, val_df, test_df]:
    dataframe["CYTOGENETIC_COMPLEXITY_GROUP"] = (
        clean_cytogenetic_complexity(
            dataframe["CYTOGENETIC_COMPLEXITY"]
        )
    )


print("\nCleaned training cytogenetic complexity:")
print(
    train_df["CYTOGENETIC_COMPLEXITY_GROUP"]
    .value_counts(dropna=False)
)


# Stop if unexpected cytogenetic values remain.
for split_name, dataframe in [
    ("training", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    if (
        dataframe["CYTOGENETIC_COMPLEXITY_GROUP"]
        == "Unrecognized"
    ).any():

        unexpected_values = dataframe.loc[
            dataframe[
                "CYTOGENETIC_COMPLEXITY_GROUP"
            ] == "Unrecognized",
            "CYTOGENETIC_COMPLEXITY"
        ].unique()

        raise ValueError(
            f"Unrecognized CYTOGENETIC_COMPLEXITY values "
            f"in {split_name}: {unexpected_values}"
        )



Cleaned training cytogenetic complexity:
CYTOGENETIC_COMPLEXITY_GROUP
1         256
0         142
2         118
4_plus     59
3          45
Name: count, dtype: int64


In [ ]:
# ------------------------------------------------------------
# 9. Check numerical skewness using TRAINING data only
# ------------------------------------------------------------

candidate_numerical_cols = [
    "AGE",
    "WBC",
    "BONE_MARROW_LEUKEMIC_BLAST_PERCENTAGE",
    "PERIPHERAL_BLASTS_PERCENTAGE"
]

candidate_numerical_cols = [
    col for col in candidate_numerical_cols
    if col in train_df.columns
]

training_skewness_before = (
    train_df[candidate_numerical_cols]
    .skew(numeric_only=True)
    .sort_values(
        key=lambda values: values.abs(),
        ascending=False
    )
)

print("\nTraining-set numerical skewness before transformation:")
print(training_skewness_before)

print(
    "\nInterpretation guideline:\n"
    "|skewness| < 0.5: approximately symmetric\n"
    "0.5–1.0: moderately skewed\n"
    "|skewness| > 1.0: strongly skewed"
)



Training-set numerical skewness before transformation:
WBC                                      2.297533
BONE_MARROW_LEUKEMIC_BLAST_PERCENTAGE   -0.643193
AGE                                      0.051862
PERIPHERAL_BLASTS_PERCENTAGE            -0.018198
dtype: float64

Interpretation guideline:
|skewness| < 0.5: approximately symmetric
0.5–1.0: moderately skewed
|skewness| > 1.0: strongly skewed


In [ ]:
# ------------------------------------------------------------
# 10. Log-transform WBC
# ------------------------------------------------------------

def add_wbc_log_feature(dataframe):
    """
    Replace strongly right-skewed WBC with log1p(WBC).

    log1p means:
        log(1 + WBC)

    It supports zero values and reduces the influence of very
    large WBC measurements without deleting clinically plausible
    extreme observations.
    """

    dataframe = dataframe.copy()

    if (dataframe["WBC"] < 0).any():
        raise ValueError(
            "Negative WBC values found. These must be reviewed "
            "before applying log1p."
        )

    dataframe["WBC_LOG1P"] = np.log1p(
        dataframe["WBC"].astype(float)
    )

    return dataframe


if APPLY_WBC_LOG_TRANSFORMATION:
    train_df = add_wbc_log_feature(train_df)
    val_df = add_wbc_log_feature(val_df)
    test_df = add_wbc_log_feature(test_df)

    print("\nWBC skewness before log1p:")
    print(train_df["WBC"].skew())

    print("WBC_LOG1P skewness after log1p:")
    print(train_df["WBC_LOG1P"].skew())



WBC skewness before log1p:
2.297532500326655
WBC_LOG1P skewness after log1p:
-0.01951706999578438


In [ ]:
# ------------------------------------------------------------
# 11. Define columns excluded from predictors
# ------------------------------------------------------------

# These variables represent the outcome or are derived from it.
OUTCOME_AND_LEAKAGE_COLS = [
    "DAYS_TO_EVENT",
    "FIRST_EVENT",
    "OS_STATUS",
    "OS_DAYS",
    "OS_MONTHS",
    "OS_EVENT"
]

DROP_FROM_PREDICTORS = [
    ID_COL,

    # Redundant with AGE:
    "AGE_IN_DAYS",

    # Original uncleaned columns:
    "RISK_GROUP",
    "CYTOGENETIC_COMPLEXITY",

] + OUTCOME_AND_LEAKAGE_COLS


# Use WBC_LOG1P instead of original WBC.
if APPLY_WBC_LOG_TRANSFORMATION:
    DROP_FROM_PREDICTORS.append("WBC")


# Optionally exclude cleaned risk group.
if not INCLUDE_RISK_GROUP:
    DROP_FROM_PREDICTORS.append("RISK_GROUP_CLEAN")


# Optionally exclude protocol.
if not INCLUDE_PROTOCOL:
    DROP_FROM_PREDICTORS.append("PROTOCOL")


DROP_FROM_PREDICTORS = list(dict.fromkeys(
    col for col in DROP_FROM_PREDICTORS
    if col in train_df.columns
))

print("\nColumns excluded from predictors:")
print(DROP_FROM_PREDICTORS)


Columns excluded from predictors:
['PATIENT_ID', 'AGE_IN_DAYS', 'RISK_GROUP', 'CYTOGENETIC_COMPLEXITY', 'DAYS_TO_EVENT', 'FIRST_EVENT', 'OS_STATUS', 'OS_DAYS', 'OS_MONTHS', 'OS_EVENT', 'WBC', 'PROTOCOL']


In [ ]:
# ------------------------------------------------------------
# 12. Separate IDs, predictors and survival outcomes
# ------------------------------------------------------------

def separate_data(dataframe):
    ids = dataframe[[ID_COL]].copy()

    survival = dataframe[
        [TIME_COL, EVENT_COL]
    ].copy()

    predictors = dataframe.drop(
        columns=DROP_FROM_PREDICTORS
    ).copy()

    return ids, predictors, survival


ids_train, X_train, y_train = separate_data(train_df)
ids_val, X_val, y_val = separate_data(val_df)
ids_test, X_test, y_test = separate_data(test_df)


# Ensure validation/test columns match training predictors.
if X_train.columns.tolist() != X_val.columns.tolist():
    raise ValueError(
        "Training and validation predictor columns differ."
    )

if X_train.columns.tolist() != X_test.columns.tolist():
    raise ValueError(
        "Training and test predictor columns differ."
    )


print("\nPredictors before encoding:")
print(X_train.columns.tolist())



Predictors before encoding:
['SEX', 'RACE', 'ETHNICITY', 'AGE', 'BONE_MARROW_LEUKEMIC_BLAST_PERCENTAGE', 'PERIPHERAL_BLASTS_PERCENTAGE', 'CNS_DISEASE', 'CHLOROMA', 'FAB', 'PRIMARY_CYTOGENETIC_CODE', 'RISK_GROUP_CLEAN', 'CYTOGENETIC_COMPLEXITY_GROUP', 'WBC_LOG1P']


In [ ]:
# ------------------------------------------------------------
# 13. Identify categorical and numerical predictors
# ------------------------------------------------------------

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "bool", "string"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()

unsupported_cols = [
    col for col in X_train.columns
    if col not in categorical_cols + numerical_cols
]

if unsupported_cols:
    raise TypeError(
        f"Unsupported predictor types found: {unsupported_cols}"
    )

print("\nNumerical predictors to standardize:")
print(numerical_cols)

print("\nCategorical predictors to one-hot encode:")
print(categorical_cols)



Numerical predictors to standardize:
['AGE', 'BONE_MARROW_LEUKEMIC_BLAST_PERCENTAGE', 'PERIPHERAL_BLASTS_PERCENTAGE', 'WBC_LOG1P']

Categorical predictors to one-hot encode:
['SEX', 'RACE', 'ETHNICITY', 'CNS_DISEASE', 'CHLOROMA', 'FAB', 'PRIMARY_CYTOGENETIC_CODE', 'RISK_GROUP_CLEAN', 'CYTOGENETIC_COMPLEXITY_GROUP']


In [ ]:
# ------------------------------------------------------------
# 14. Preprocessing pipelines
# ------------------------------------------------------------

# Numerical pipeline:
# 1. Median imputation is included as a safety fallback.
# 2. Standardize using training mean and standard deviation.
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# One-hot encoder compatibility for different sklearn versions.
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
        drop=None
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False,
        drop=None
    )


# Categorical pipeline:
# Missing categories receive an explicit 'Missing' category.
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Missing"
            )
        ),
        (
            "encoder",
            encoder
        )
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numeric_pipeline,
            numerical_cols
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_cols
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)


In [ ]:
# ------------------------------------------------------------
# 15. Fit ONLY on the training set
# ------------------------------------------------------------

X_train_array = preprocessor.fit_transform(X_train)

# Do not fit again on validation or test.
X_val_array = preprocessor.transform(X_val)
X_test_array = preprocessor.transform(X_test)


# ------------------------------------------------------------
# 16. Restore transformed feature names
# ------------------------------------------------------------

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_array,
    columns=feature_names,
    index=X_train.index
)

X_val_processed = pd.DataFrame(
    X_val_array,
    columns=feature_names,
    index=X_val.index
)

X_test_processed = pd.DataFrame(
    X_test_array,
    columns=feature_names,
    index=X_test.index
)


# ------------------------------------------------------------
# 17. Validate transformed data
# ------------------------------------------------------------

for split_name, predictors in [
    ("Training", X_train_processed),
    ("Validation", X_val_processed),
    ("Test", X_test_processed)
]:
    if predictors.isna().any().any():
        raise ValueError(
            f"Missing values remain in processed {split_name} data."
        )

    if np.isinf(predictors.to_numpy()).any():
        raise ValueError(
            f"Infinite values found in processed {split_name} data."
        )


assert (
    X_train_processed.columns.tolist()
    == X_val_processed.columns.tolist()
)

assert (
    X_train_processed.columns.tolist()
    == X_test_processed.columns.tolist()
)


print("\nProcessed dataset shapes:")
print("Training:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

print("\nNumber of processed predictors:")
print(len(feature_names))

print("\nProcessed feature names:")
print(feature_names[:40])


Processed dataset shapes:
Training: (620, 40)
Validation: (177, 40)
Test: (89, 40)

Number of processed predictors:
40

Processed feature names:
['AGE' 'BONE_MARROW_LEUKEMIC_BLAST_PERCENTAGE'
 'PERIPHERAL_BLASTS_PERCENTAGE' 'WBC_LOG1P' 'SEX_Female' 'SEX_Male'
 'RACE_American Indian or Alaska Native' 'RACE_Asian'
 'RACE_Black or African American'
 'RACE_Native Hawaiian or other Pacific Islander' 'RACE_Other'
 'RACE_White' 'ETHNICITY_Hispanic or Latino'
 'ETHNICITY_Not Hispanic or Latino' 'CNS_DISEASE_No' 'CNS_DISEASE_Yes'
 'CHLOROMA_No' 'CHLOROMA_Yes' 'FAB_M0' 'FAB_M1' 'FAB_M2' 'FAB_M4' 'FAB_M5'
 'FAB_M6' 'FAB_M7' 'FAB_NOS' 'PRIMARY_CYTOGENETIC_CODE_MLL'
 'PRIMARY_CYTOGENETIC_CODE_Normal' 'PRIMARY_CYTOGENETIC_CODE_Other'
 'PRIMARY_CYTOGENETIC_CODE_inv(16)' 'PRIMARY_CYTOGENETIC_CODE_t(8;21)'
 'RISK_GROUP_CLEAN_High' 'RISK_GROUP_CLEAN_Low' 'RISK_GROUP_CLEAN_Missing'
 'RISK_GROUP_CLEAN_Standard' 'CYTOGENETIC_COMPLEXITY_GROUP_0'
 'CYTOGENETIC_COMPLEXITY_GROUP_1' 'CYTOGENETIC_COMPLEXITY_GRO

In [ ]:
# ------------------------------------------------------------
# 18. Combine ID, predictors and survival outcomes
# ------------------------------------------------------------

def create_model_ready_file(
    ids,
    predictors,
    survival
):
    ids = ids.reset_index(drop=True)
    predictors = predictors.reset_index(drop=True)
    survival = survival.reset_index(drop=True)

    return pd.concat(
        [ids, predictors, survival],
        axis=1
    )


train_model_ready = create_model_ready_file(
    ids_train,
    X_train_processed,
    y_train
)

val_model_ready = create_model_ready_file(
    ids_val,
    X_val_processed,
    y_val
)

test_model_ready = create_model_ready_file(
    ids_test,
    X_test_processed,
    y_test
)


# ------------------------------------------------------------
# 19. Save model-ready CSV files
# ------------------------------------------------------------

train_output_path = (
    OUTPUT_DIR / "train_survival_model_ready.csv"
)

val_output_path = (
    OUTPUT_DIR / "validation_survival_model_ready.csv"
)

test_output_path = (
    OUTPUT_DIR / "test_survival_model_ready.csv"
)

train_model_ready.to_csv(
    train_output_path,
    index=False
)

val_model_ready.to_csv(
    val_output_path,
    index=False
)

test_model_ready.to_csv(
    test_output_path,
    index=False
)


# ------------------------------------------------------------
# 20. Save fitted preprocessor
# ------------------------------------------------------------

preprocessor_path = (
    OUTPUT_DIR / "survival_preprocessor.joblib"
)

joblib.dump(
    preprocessor,
    preprocessor_path
)


# ------------------------------------------------------------
# 21. Save preprocessing metadata
# ------------------------------------------------------------

metadata = {
    "id_column": ID_COL,
    "survival_time_column": TIME_COL,
    "survival_event_column": EVENT_COL,

    "include_risk_group": INCLUDE_RISK_GROUP,
    "include_protocol": INCLUDE_PROTOCOL,
    "wbc_log_transformation": APPLY_WBC_LOG_TRANSFORMATION,

    "risk_group_mapping": {
        "10": "Low",
        "30": "High",
        "missing": "Missing"
    },

    "cytogenetic_complexity_mapping": {
        "0": "0",
        "1": "1",
        "2": "2",
        "3": "3",
        "4_or_greater": "4_plus",
        "more_than_6": "4_plus"
    },

    "training_skewness_before_transformation": {
        col: float(value)
        for col, value
        in training_skewness_before.items()
    },

    "columns_dropped_from_predictors":
        DROP_FROM_PREDICTORS,

    "original_predictor_columns":
        X_train.columns.tolist(),

    "numerical_columns_scaled":
        numerical_cols,

    "categorical_columns_encoded":
        categorical_cols,

    "processed_feature_names":
        feature_names.tolist(),

    "n_train":
        len(train_model_ready),

    "n_validation":
        len(val_model_ready),

    "n_test":
        len(test_model_ready)
}


metadata_path = (
    OUTPUT_DIR / "survival_preprocessing_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False
    )



In [ ]:
# ------------------------------------------------------------
# 22. Final report
# ------------------------------------------------------------

print("\nPreprocessing completed successfully.")
print("\nSaved files:")

for output_path in [
    train_output_path,
    val_output_path,
    test_output_path,
    preprocessor_path,
    metadata_path
]:
    print("-", output_path)


Preprocessing completed successfully.

Saved files:
- /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Patient/survival_model_ready/train_survival_model_ready.csv
- /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Patient/survival_model_ready/validation_survival_model_ready.csv
- /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Patient/survival_model_ready/test_survival_model_ready.csv
- /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Patient/survival_model_ready/survival_preprocessor.joblib
- /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Patient/survival_model_ready/survival_preprocessing_metadata.json
